[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/03_sandbox_harness/03_sandbox_harness.ipynb)

# 03 · 沙箱、轨迹与 agent harness —— 动手篇

**纯 CPU / 纯标准库**（`subprocess` + `tempfile` + `json`），不需要任何模型或 API key。
本模块的主角是**评测基础设施本身**，所以我们用一个脚本化的 LLM 替身驱动 agent 循环——这让每个机制都确定、可断言。

本 notebook 从零搭一个最小但五脏俱全的 harness：

1. **`Sandbox`** —— 隔离工作目录 + 步级超时 kill + 输出截断 + `reset()` 可重置（讲解 §1–2 的 subprocess 级隔离）；
2. **`TrajectoryLogger` / `LoggedSandbox`** —— JSONL 事件流，每个动作自动落盘（讲解 §4 轨迹即证据）；
3. **接入模块 02 的 ReAct 循环** —— 给 agent 一个 `run_python` 工具，完成真实任务并录下完整轨迹；
4. **`replay()`** —— 录制-重放测试法：用日志里的 LLM 输出重跑循环，assert 与原始一致（讲解 §5）；
5. ✏️ 三道练习：输出截断器、成本聚合（为模块 06 打底）、命令白名单（为模块 08 埋线）。

> 工程视角：METR 的 Vivaria 与 UK AISI 的 Inspect 做的就是这套东西的生产级版本 [METR 2024; UK AISI 2024]。接口同构，隔离级别从 subprocess 换成 container/VM 而已。

In [ ]:
import json, os, shutil, subprocess, sys, tempfile, time
from pathlib import Path


class Sandbox:
    '''最小沙箱：隔离工作目录 + 超时 kill + 输出截断 + 可重置。

    隔离级别 = subprocess（防事故，不防恶意）。换成 container/VM 时
    这套接口（run / write_file / read_file / reset）保持不变。
    '''

    def __init__(self, max_bytes=4096, timeout=10.0):
        self.max_bytes = max_bytes        # 单条输出的字节上限（防输出爆炸）
        self.default_timeout = timeout    # 步级超时（秒）
        self.root = Path(tempfile.mkdtemp(prefix="sbx_"))

    def write_file(self, relpath, content):
        p = self.root / relpath
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content, encoding="utf-8")
        return relpath

    def read_file(self, relpath):
        return (self.root / relpath).read_text(encoding="utf-8")

    def run(self, cmd, timeout=None):
        '''在沙箱目录中执行命令。cmd 是参数列表（不经 shell，避免注入）。'''
        t0 = time.monotonic()
        try:
            proc = subprocess.run(cmd, cwd=self.root, capture_output=True,
                                  timeout=timeout or self.default_timeout)
            rc, out, err, timed_out = proc.returncode, proc.stdout, proc.stderr, False
        except subprocess.TimeoutExpired as e:   # 超时：subprocess.run 已 kill 子进程
            rc, out, err, timed_out = None, e.stdout or b"", e.stderr or b"", True
        return {"returncode": rc,
                "stdout": self._truncate(out),
                "stderr": self._truncate(err),
                "timed_out": timed_out,
                "wall_time_s": round(time.monotonic() - t0, 4)}

    def _truncate(self, raw):
        if len(raw) > self.max_bytes:
            omitted = len(raw) - self.max_bytes
            return (raw[:self.max_bytes].decode("utf-8", errors="replace")
                    + f"\n...[truncated {omitted} bytes]")
        return raw.decode("utf-8", errors="replace")

    def reset(self):
        '''整个目录删掉重建 —— 评测要求每个 run 从同一初始状态 E0 出发。'''
        shutil.rmtree(self.root, ignore_errors=True)
        self.root = Path(tempfile.mkdtemp(prefix="sbx_"))

    def close(self):
        shutil.rmtree(self.root, ignore_errors=True)


print("Sandbox 就绪（工作目录隔离在 tempdir，宿主项目文件不受影响）")

In [ ]:
sbx = Sandbox(max_bytes=2048, timeout=5.0)

# --- 1) 写文件 → 执行 Python 脚本 → 读结果 ---------------------------------
script = '''
result = sum(range(100))
with open("out.txt", "w") as f:
    f.write(str(result))
print("computed:", result)
'''
sbx.write_file("compute.py", script)
r = sbx.run([sys.executable, "compute.py"])
print(r["stdout"].strip(), "| returncode:", r["returncode"], "| 耗时:", r["wall_time_s"], "s")
assert sbx.read_file("out.txt") == "4950"

# --- 2) 步级超时：死循环脚本被 kill，harness 不挂死 -------------------------
sbx.write_file("hang.py", "while True: pass")
r = sbx.run([sys.executable, "hang.py"], timeout=1.0)
print("超时演示: timed_out =", r["timed_out"], "| returncode =", r["returncode"])
assert r["timed_out"] and r["returncode"] is None

# --- 3) 输出爆炸防护：脚本打印 1MB，观察截断生效 -----------------------------
sbx.write_file("flood.py", "print('A' * 1_000_000)")
r = sbx.run([sys.executable, "flood.py"])
print("flood stdout 长度:", len(r["stdout"]), "字符（原始 ~1MB）")
print("截断标记:", repr(r["stdout"][-40:]))
assert "truncated" in r["stdout"] and len(r["stdout"]) < 5000

# --- 4) reset：环境回到初始状态，run 之间互不污染 ----------------------------
sbx.reset()
assert list(sbx.root.iterdir()) == []
print("reset 后目录为空 ✅ —— 上一个 run 的所有痕迹（脚本、out.txt）都不存在了")

## 轨迹即证据：JSONL 事件流

agent 评测的原始产出是**轨迹**，分数只是它的汇总统计量。我们用追加写的 JSONL：每行一个事件、写完即落盘（崩溃最多丢一行）。最小事件模型：

| event_type | payload 必记字段 | 用途 |
|---|---|---|
| `llm_call` | 完整输入规模、**完整输出**、采样参数、时延 | 重放素材；审计模型看到/说了什么 |
| `tool_call` | 工具名、完整参数 | 还原行动序列；安全审查 |
| `observation` | stdout/stderr、退出码、是否截断/超时 | 还原 agent 的信息输入 |
| `score` | 最终答案、状态、评分依据 | 结果本体；支持事后重评分 |

`LoggedSandbox` 用组合（而非继承）把日志横切进沙箱——agent 代码完全不知道自己被记录，这正是 harness 的职责边界：**记录属于基础设施，不属于被测对象**。

In [ ]:
class TrajectoryLogger:
    '''追加写 JSONL 事件流：每行一个事件，写完即落盘。'''

    def __init__(self, path):
        self.path = Path(path)
        self.path.write_text("", encoding="utf-8")   # 清空旧日志
        self.step = 0

    def log(self, event_type, payload, duration_s=None):
        self.step += 1
        event = {"step": self.step, "ts": round(time.time(), 3),
                 "event_type": event_type, "duration_s": duration_s,
                 "payload": payload}
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(event, ensure_ascii=False) + "\n")
        return event

    def events(self):
        lines = self.path.read_text(encoding="utf-8").splitlines()
        return [json.loads(line) for line in lines if line.strip()]


class LoggedSandbox:
    '''包一层 Sandbox：每次 write_file / run 自动写入轨迹。'''

    def __init__(self, sandbox, logger):
        self.sandbox, self.logger = sandbox, logger

    def write_file(self, relpath, content):
        self.logger.log("tool_call", {"tool": "write_file", "path": relpath,
                                      "n_chars": len(content)})
        return self.sandbox.write_file(relpath, content)

    def run(self, cmd, timeout=None):
        self.logger.log("tool_call", {"tool": "sandbox.run", "cmd": list(cmd)})
        result = self.sandbox.run(cmd, timeout=timeout)
        self.logger.log("observation",
                        {"returncode": result["returncode"],
                         "stdout": result["stdout"], "stderr": result["stderr"],
                         "timed_out": result["timed_out"]},
                        duration_s=result["wall_time_s"])
        return result


# 冒烟测试
_sbx = Sandbox()
_log = TrajectoryLogger("smoke.jsonl")
_ls = LoggedSandbox(_sbx, _log)
_ls.write_file("t.py", "print(1 + 1)")
_ls.run([sys.executable, "t.py"])
for e in _log.events():
    print(f"step {e['step']}  {e['event_type']:<12} {str(e['payload'])[:60]}")
_sbx.close()

## 接入模块 02 的 ReAct 循环

下面把模块 02 的 Thought → Action → Observation 循环接到沙箱上：agent 唯一的工具是 `run_python`（代码经 `LoggedSandbox` 写入并执行）。任务：**写脚本统计一段文本的词频并输出 top3**。

LLM 用 `ScriptedLLM` 替身（按固定顺序吐出预写回复）——真实 harness 中这里是一次 API 调用，但 harness 的其余部分**一行都不用改**。这种可替换性正是第 5 节重放的基础：`ScriptedLLM` / `ReplayLLM` / 真模型对循环来说是同一个接口。

In [ ]:
SYSTEM_PROMPT = (
    "你是一个 ReAct agent。可用工具:\n"
    "  run_python —— 在沙箱里执行 Python 代码\n"
    "输出格式（二选一）:\n"
    "  Thought: <推理>\n  Action: run_python\n  ```python\n  <代码>\n  ```\n"
    "或:\n  Thought: <推理>\n  Final Answer: <答案>"
)


def parse_action(response):
    '''模型输出 → ("final", 答案) | ("run_python", 代码) | ("invalid", None)'''
    if "Final Answer:" in response:
        return "final", response.split("Final Answer:", 1)[1].strip()
    if "```python" in response:
        code_str = response.split("```python", 1)[1].split("```", 1)[0].strip()
        return "run_python", code_str
    return "invalid", None


def run_agent(llm, logged_sbx, logger, task, max_steps=6):
    '''最小 harness 主循环：LLM ↔ 沙箱，全程落轨迹。'''
    logger.log("task_start", {"task": task})
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": task}]
    for _ in range(max_steps):
        t0 = time.monotonic()
        response = llm.chat(messages)
        logger.log("llm_call",
                   {"n_messages": len(messages), "response": response},
                   duration_s=round(time.monotonic() - t0, 4))
        messages.append({"role": "assistant", "content": response})
        kind, arg = parse_action(response)
        if kind == "final":
            logger.log("score", {"final_answer": arg, "status": "completed"})
            return arg
        elif kind == "run_python":
            logged_sbx.write_file("script.py", arg)
            r = logged_sbx.run([sys.executable, "script.py"])
            obs = (f"returncode={r['returncode']}, timed_out={r['timed_out']}\n"
                   f"stdout:\n{r['stdout']}\nstderr:\n{r['stderr']}")
            messages.append({"role": "user", "content": "Observation:\n" + obs})
        else:
            messages.append({"role": "user",
                             "content": "Observation: 输出格式无法解析，请按规定格式重试。"})
    logger.log("score", {"final_answer": None, "status": "max_steps_exceeded"})
    return None


class ScriptedLLM:
    '''脚本化 LLM 替身：按固定顺序吐出预写回复（CPU-only，确定性）。'''
    def __init__(self, responses):
        self.responses, self.i = list(responses), 0
    def chat(self, messages):
        resp = self.responses[self.i]
        self.i += 1
        return resp


TEXT = "the quick brown fox jumps over the lazy dog the fox"
TASK = ("沙箱工作目录里有 text.txt。写一个 Python 脚本统计其中的词频，"
        "并以 word:count 逗号分隔的格式输出出现次数最多的 top3。")

SCRIPTED = [
    '''Thought: 读 text.txt，用 collections.Counter 统计词频，输出 top3。
Action: run_python
```python
import collections, re
text = open("text.txt", encoding="utf-8").read().lower()
words = re.findall(r"[a-z]+", text)
top3 = collections.Counter(words).most_common(3)
print(",".join(f"{w}:{c}" for w, c in top3))
```''',
    '''Thought: 脚本成功输出了 top3 词频，作为最终答案提交。
Final Answer: the:3,fox:2,quick:1''',
]

sbx = Sandbox(max_bytes=4096, timeout=10.0)
logger = TrajectoryLogger("trajectory.jsonl")
lsbx = LoggedSandbox(sbx, logger)
lsbx.write_file("text.txt", TEXT)   # 任务环境布置（对应 Task Standard 的 start()）

answer = run_agent(ScriptedLLM(SCRIPTED), lsbx, logger, TASK)
print("最终答案:", answer)
print("\n完整轨迹事件序列:")
for e in logger.events():
    print(f"  step {e['step']:>2}  {e['event_type']:<12} {str(e['payload'])[:68]}")

## 录制-重放（record & replay）

轨迹里已经录下了模型的每一次输出。**重放** = 用 `ReplayLLM` 按序复读这些输出、重新执行整条循环。若 harness 与环境都是确定的，重放必须逐事件复现原始轨迹——

- 任何 observation 失配 ⇒ 环境有非确定性（时间戳/随机数/外部 API）；
- `ReplayLLM` 越界（多调了一次）⇒ harness 控制流变了；
- 最终答案不同 ⇒ 解析/评分逻辑变了。

这就是 harness 的**免费回归测试**：录一次（贵），之后改解析器、改日志格式都在重放上验证（毫秒级、零 token 成本）。注意环境布置（`text.txt`）必须与录制时一致——生产系统中由"镜像固定 digest"保证。

In [ ]:
class ReplayLLM:
    '''重放替身：按序复读轨迹中录制的 LLM 输出。'''
    def __init__(self, recorded_responses):
        self.responses, self.i = list(recorded_responses), 0
    def chat(self, messages):
        if self.i >= len(self.responses):
            raise RuntimeError("重放越界：harness 比录制时多调了一次 LLM（控制流变更）")
        resp = self.responses[self.i]
        self.i += 1
        return resp


def replay(log_path):
    '''读取轨迹中的 LLM 输出序列，用 ReplayLLM 重跑循环，返回 (重放答案, 原始答案)。'''
    events = [json.loads(line)
              for line in Path(log_path).read_text(encoding="utf-8").splitlines()
              if line.strip()]
    recorded = [e["payload"]["response"] for e in events if e["event_type"] == "llm_call"]
    original = next(e["payload"]["final_answer"] for e in events if e["event_type"] == "score")

    sbx2 = Sandbox(max_bytes=4096, timeout=10.0)
    logger2 = TrajectoryLogger("replay.jsonl")
    lsbx2 = LoggedSandbox(sbx2, logger2)
    lsbx2.write_file("text.txt", TEXT)   # 环境布置必须与录制时一致
    replayed = run_agent(ReplayLLM(recorded), lsbx2, logger2, TASK)
    sbx2.close()
    return replayed, original


replayed_answer, original_answer = replay("trajectory.jsonl")
print("原始答案:", original_answer)
print("重放答案:", replayed_answer)
assert replayed_answer == original_answer, "重放不一致 → 存在非确定性或 harness 行为变更"


def observations(path):
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        e = json.loads(line)
        if e["event_type"] == "observation":
            out.append(e["payload"]["stdout"])
    return out

assert observations("trajectory.jsonl") == observations("replay.jsonl")
print("✅ 重放一致（最终答案 + 逐 observation）—— harness 行为可复现")
sbx.close()

## ✏️ 练习 1：head_tail 截断器

`Sandbox._truncate` 只保留开头——但调试时**结尾往往更有信息量**（traceback 在最后）。实现：

```
truncate_output(text, max_bytes, keep="head_tail") -> str
```

要求：
- `text` 的 UTF-8 字节数 ≤ `max_bytes` 时原样返回；
- 否则保留头部约 `max_bytes//2` 字节 + 尾部剩余预算字节，中间插入标记 `...[omitted N bytes]...`（`N` = 被省略的字节数）；
- 除标记外保留的内容 ≤ `max_bytes` 字节。

**提示**：先 `raw = text.encode("utf-8")`，对 `raw` 切片后 `decode("utf-8", errors="replace")`（防止从多字节字符中间切断报错）。10 行以内可完成。

In [ ]:
def truncate_output(text, max_bytes, keep="head_tail"):
    '''截断 text 到 ~max_bytes 字节（UTF-8 计），保留头尾，中间放省略标记。'''
    raw = text.encode("utf-8")
    if len(raw) <= max_bytes:
        return text
    # TODO: 1) omitted = 被省略的字节数
    #       2) head = 前 max_bytes//2 字节，tail = 后 max_bytes - max_bytes//2 字节
    #       3) 返回 head + f"\n...[omitted {omitted} bytes]...\n" + tail
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
assert truncate_output("hello", 100) == "hello"               # 不超限：原样返回

long_text = "A" * 500 + "MIDDLE" + "B" * 500                  # 1006 字节
out = truncate_output(long_text, 200)
assert "omitted" in out, "缺少省略标记"
assert "806" in out, "应准确报告省略的字节数 (1006-200=806)"
assert out.startswith("A" * 50), "头部应被保留"
assert out.endswith("B" * 50), "尾部应被保留"
assert len(out.encode("utf-8")) <= 200 + 64, "总长应受控（标记预算 64 字节）"
print("✅ 练习 1 通过")

## ✏️ 练习 2：成本聚合 `cost_summary`

资源计量是成本感知评测（模块 06）的地基。从 JSONL 轨迹聚合出：

```
cost_summary(log_path) -> {"n_llm_calls": int, "n_tool_calls": int,
                           "total_duration_s": float, "est_tokens": int}
```

- `n_llm_calls` / `n_tool_calls`：对应 event_type 的事件数；
- `total_duration_s`：所有事件 `duration_s` 之和（`None` 按 0 计）；
- `est_tokens`：对每个 `llm_call`，按 `len(payload["response"]) // 4` 粗估 token 数（生产中用 API 返回的 usage 字段或 tokenizer 精确计数）。

**提示**：逐行 `json.loads`，按 `event_type` 分支累加。`event.get("duration_s") or 0.0` 可同时处理缺失与 `None`。

In [ ]:
def cost_summary(log_path):
    '''从 JSONL 轨迹聚合资源用量。'''
    summary = {"n_llm_calls": 0, "n_tool_calls": 0,
               "total_duration_s": 0.0, "est_tokens": 0}
    for line in Path(log_path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        event = json.loads(line)
        # TODO: 按 event["event_type"] 更新 summary 的四个字段
        raise NotImplementedError
    summary["total_duration_s"] = round(summary["total_duration_s"], 4)
    return summary

In [ ]:
# ---- 练习 2 自测 ----
_test_events = [
    {"step": 1, "event_type": "llm_call", "duration_s": 0.5,
     "payload": {"response": "x" * 40}},
    {"step": 2, "event_type": "tool_call", "duration_s": 1.5,
     "payload": {"tool": "sandbox.run", "cmd": ["echo", "hi"]}},
    {"step": 3, "event_type": "llm_call", "duration_s": 0.25,
     "payload": {"response": "y" * 80}},
    {"step": 4, "event_type": "observation", "duration_s": None,
     "payload": {"stdout": "hi"}},
]
with open("test_cost.jsonl", "w", encoding="utf-8") as f:
    for e in _test_events:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

s = cost_summary("test_cost.jsonl")
assert s["n_llm_calls"] == 2, s
assert s["n_tool_calls"] == 1, s
assert abs(s["total_duration_s"] - 2.25) < 1e-9, s
assert s["est_tokens"] == 30, s        # 40//4 + 80//4

s2 = cost_summary("trajectory.jsonl")  # 在真轨迹上也能跑
assert s2["n_llm_calls"] == 2 and s2["n_tool_calls"] >= 2, s2
print("✅ 练习 2 通过 | 真实轨迹聚合:", s2)

## ✏️ 练习 3：命令白名单 `SafeSandbox`

为模块 08（agent 安全）埋线：评测危险能力时，harness 常对 agent 的行动空间做**白名单约束**，越界动作要"拒绝 + 留痕"——拒绝保护环境，留痕本身就是评测信号（模型*试图*做什么，与它*做成*什么同样重要）。

继承 `Sandbox`，新增 `allowed_commands` 参数（程序名集合，按 `os.path.basename(cmd[0])` 比较；`None` = 不限制）：

- 白名单内 → 正常执行，结果加 `"blocked": False`；
- 白名单外 → **不执行**，把违规事件 `{"cmd": ..., "ts": ...}` 追加进 `self.violations`，返回
  `{"returncode": None, "stdout": "", "stderr": "BLOCKED: ...", "timed_out": False, "wall_time_s": 0.0, "blocked": True}`。

**提示**：只需覆写 `run()`，约 10 行。

In [ ]:
class SafeSandbox(Sandbox):
    '''带命令白名单的沙箱：白名单外的命令拒绝执行并记录违规事件。'''

    def __init__(self, allowed_commands=None, **kwargs):
        super().__init__(**kwargs)
        self.allowed_commands = allowed_commands
        self.violations = []   # 违规事件列表

    def run(self, cmd, timeout=None):
        # TODO: 1) prog = os.path.basename(cmd[0])
        #       2) 若 allowed_commands 不为 None 且 prog 不在其中:
        #          记录违规事件到 self.violations，返回 blocked=True 的结果 dict（不执行）
        #       3) 否则调用 super().run(...)，给结果加 "blocked": False
        raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
py = os.path.basename(sys.executable)              # 如 "python3"
safe = SafeSandbox(allowed_commands={py}, max_bytes=2048, timeout=5.0)

safe.write_file("ok.py", "print('ok')")
r1 = safe.run([sys.executable, "ok.py"])           # 白名单内 → 正常执行
assert r1["blocked"] is False and r1["stdout"].strip() == "ok", r1

r2 = safe.run(["rm", "-rf", "/tmp/nonexistent"])   # 白名单外 → 拒绝且不执行
assert r2["blocked"] is True and r2["returncode"] is None, r2
assert "BLOCKED" in r2["stderr"], r2
assert len(safe.violations) == 1 and safe.violations[0]["cmd"][0] == "rm"

safe.close()
print("✅ 练习 3 通过 —— 模块 08 会把'行动白名单 + 违规留痕'扩展成完整的 AI control 策略")

## 📖 参考答案

先自己做，再对照。每个参考实现都能让上面的自测 cell 全部通过。

In [ ]:
# 参考答案 · 练习 1 —— 先自己做，再对照
def truncate_output(text, max_bytes, keep="head_tail"):
    raw = text.encode("utf-8")
    if len(raw) <= max_bytes:
        return text
    omitted = len(raw) - max_bytes
    if keep == "head_tail":
        half = max_bytes // 2
        head = raw[:half].decode("utf-8", errors="replace")
        tail = raw[len(raw) - (max_bytes - half):].decode("utf-8", errors="replace")
        return head + f"\n...[omitted {omitted} bytes]...\n" + tail
    head = raw[:max_bytes].decode("utf-8", errors="replace")   # keep="head"
    return head + f"\n...[omitted {omitted} bytes]"


_out = truncate_output("A" * 500 + "MIDDLE" + "B" * 500, 200)
assert "806" in _out and len(_out.encode("utf-8")) <= 264
print("参考答案 1 自检通过:", _out[:8], "...", _out[-8:])

In [ ]:
# 参考答案 · 练习 2 —— 先自己做，再对照
def cost_summary(log_path):
    summary = {"n_llm_calls": 0, "n_tool_calls": 0,
               "total_duration_s": 0.0, "est_tokens": 0}
    for line in Path(log_path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        event = json.loads(line)
        etype = event["event_type"]
        if etype == "llm_call":
            summary["n_llm_calls"] += 1
            summary["est_tokens"] += len(event["payload"]["response"]) // 4
        elif etype == "tool_call":
            summary["n_tool_calls"] += 1
        summary["total_duration_s"] += event.get("duration_s") or 0.0
    summary["total_duration_s"] = round(summary["total_duration_s"], 4)
    return summary


print("参考答案 2 自检:", cost_summary("test_cost.jsonl"))

In [ ]:
# 参考答案 · 练习 3 —— 先自己做，再对照
class SafeSandbox(Sandbox):
    def __init__(self, allowed_commands=None, **kwargs):
        super().__init__(**kwargs)
        self.allowed_commands = allowed_commands
        self.violations = []

    def run(self, cmd, timeout=None):
        prog = os.path.basename(cmd[0])
        if self.allowed_commands is not None and prog not in self.allowed_commands:
            self.violations.append({"cmd": list(cmd), "ts": round(time.time(), 3)})
            return {"returncode": None, "stdout": "",
                    "stderr": f"BLOCKED: '{prog}' 不在 allowed_commands 白名单中",
                    "timed_out": False, "wall_time_s": 0.0, "blocked": True}
        result = super().run(cmd, timeout=timeout)
        result["blocked"] = False
        return result


_s = SafeSandbox(allowed_commands={os.path.basename(sys.executable)})
assert _s.run(["curl", "http://example.com"])["blocked"] is True
assert len(_s.violations) == 1
_s.close()
print("参考答案 3 自检通过")

## 小结

你已经拥有一个完整的迷你评测栈：

- **`Sandbox`**：隔离 + 步级超时 + 输出截断 + `reset()` —— 评测三要求（隔离/可重置/可复现）的 subprocess 级实现；
- **`TrajectoryLogger` + `LoggedSandbox`**：JSONL 事件流，进出模型与沙箱的每个事件都可审计 —— 轨迹即证据；
- **`run_agent`**：模块 02 的 ReAct 循环跑在沙箱里，LLM 接口可替换（Scripted / Replay / 真模型）；
- **`replay()`**：录制-重放回归测试 —— 不花一个 token 验证 harness 行为没变；
- 练习产物：head_tail 截断器、`cost_summary`（→ 模块 06 成本感知评测）、`SafeSandbox` 白名单（→ 模块 08 AI control）。

**下一章（04 · 编码 Agent 解剖：SWE-bench）**把这套骨架放大到生产尺度：SWE-bench 的"一题一 Docker 容器"就是 `Sandbox` 的 container 级版本，`score()` 变成跑真实仓库的测试套件，轨迹变成数千步的真实编码过程 [Jimenez 2023; Yang 2024]。

---
## 🎯 真实数据胶囊题：真实 MBPP 题上的代码执行 harness

编码 agent 的评测核心是：把候选代码跑起来、对测试用例判通过与否。用真实 MBPP（Python 编程题+测试），实现一个执行 harness，验证官方参考解能通过它自己的测试。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

probs=mbpp(60)
ex=probs[0]
print("真实 MBPP 题:", ex["text"][:70])
print("测试用例:", ex["test_list"][0])

**练习**：实现 `run_tests(code, test_list)`：在一个独立命名空间里 exec(code)，再逐条 exec 每个 assert 测试；全过返回 True，任一失败/异常返回 False。

In [ ]:
def run_tests(code, test_list):
    # TODO: ns={}; exec(code,ns); 对每个 test exec(test,ns)；有异常返回 False，全过 True
    raise NotImplementedError


In [ ]:
# 自测：官方参考解应通过自己的测试
passed=sum(run_tests(p["code"], p["test_list"]) for p in probs[:40])
assert passed/40 > 0.85, f"参考解应通过>85%自身测试, 得到{passed}/40"
# 故意写错的代码应判 fail
assert run_tests("def f(): return 0", ["assert f()==1"])==False
print(f"harness: {passed}/40 真实参考解通过自身测试 ✓")


### 📖 参考答案

In [ ]:
def run_tests(code, test_list):
    ns={}
    try:
        exec(code, ns)
        for t in test_list: exec(t, ns)
        return True
    except Exception:
        return False
print("✓ 执行 + 断言判定 = 编码 agent 评测的地基(SWE-bench 同理)")